In [ ]:
!git clone https://github.com/CryAndRRich/codapath.git

In [ ]:
%cd /kaggle/working/codapath
CODAPATH = "/kaggle/working/codapath"

In [ ]:
!pip install -r requirements.txt
!pip install -U huggingface_hub hf-transfer

In [ ]:
import os
from huggingface_hub import login, snapshot_download

login("YOUR_HUGGINGFACE_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("Downloading vinid/plip...")
snapshot_download(repo_id="vinid/plip")

print("Downloading BiomedCLIP...")
snapshot_download(repo_id="microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")

In [ ]:
import os
import sys
import yaml
import torch

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if CODAPATH not in sys.path:
    sys.path.append(CODAPATH)

In [ ]:
from ablations.run import main
from ablations.model import inspect_sampler_alignment
from load_data import get_data_loaders

CONFIG_PATH = "config/config.yaml"

# Adjust only these values.
DATASET = "pathmnist"
ABLATION_APPROACH = "plip_only"

PATHMNIST_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/pathmnist_224.npz"
HISTOSET_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/HistoSet-5x14/HistoSet-5x14"
SKINTISSUE_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/SkinTissue/SkinTissue/tiles"

DATA_PATHS = {
    "pathmnist": PATHMNIST_PATH,
    "histoset": HISTOSET_PATH,
    "skintissue": SKINTISSUE_PATH,
}

with open(CONFIG_PATH, "r", encoding="utf-8") as file_obj:
    config = yaml.safe_load(file_obj)

hyper = config["hyperparameters"]
dataset_info = config["datasets"][DATASET]
device = torch.device(config["device"])

In [ ]:
train_loader, _, class_names = get_data_loaders(DATA_PATHS[DATASET], config["random_seed"], verbose=True)
alignment_report = inspect_sampler_alignment(
    dataloader=train_loader,
    class_descriptions=dataset_info["descriptions"],
    prompt_templates=config["prompt_templates"],
    class_names=class_names,
    device=device,
    ablation_approach=ABLATION_APPROACH,
    sampler_name="codapath",
)
print(alignment_report)
assert alignment_report["feature_width_match"], alignment_report

In [ ]:
main(
    data_path=DATA_PATHS[DATASET],
    sampler_name="codapath",
    num_classes=dataset_info["num_classes"],
    cumulative_budget=config["cumulative_budget"],
    data_descriptions=dataset_info["descriptions"],
    prompt_templates=config["prompt_templates"],
    rank_lora=hyper["rank_lora"],
    num_epochs=hyper["num_epochs"],
    learn_rate=hyper["learning_rate"],
    alpha=hyper["alpha"],
    device=device,
    random_seed=config["random_seed"],
    save_dir=f"ablations/checkpoints/{DATASET}",
    ablation_approach=ABLATION_APPROACH,
    verbose=True,
)